In [1]:
import pandas as pd
import numpy as np
from deep_translator import GoogleTranslator
import time
import os
from datetime import datetime
import re

class CSVTranslator:
    """CSV文件翻译器 - 专门处理CSV文件的翻译任务"""
    
    def __init__(self, source_lang='pt', target_lang='zh-CN'):
        """
        初始化翻译器
        
        参数:
            source_lang: 源语言代码 (默认: 'pt' - 葡萄牙语)
            target_lang: 目标语言代码 (默认: 'zh-CN' - 简体中文)
        """
        self.source_lang = source_lang
        self.target_lang = target_lang
        self.translator = GoogleTranslator(source=source_lang, target=target_lang)
        self.stats = {
            'total_rows': 0,
            'filtered_rows': 0,
            'translated_items': 0,
            'failed_items': 0,
            'start_time': None,
            'end_time': None
        }
    
    def load_and_filter_csv(self, csv_path, offensive_label=1):
        """
        加载CSV文件并过滤数据
        
        参数:
            csv_path: CSV文件路径
            offensive_label: 过滤条件，默认筛选offensive_label=1的数据
            
        返回:
            DataFrame: 过滤后的数据
        """
        try:
            # 读取CSV文件
            df = pd.read_csv(csv_path)
            self.stats['total_rows'] = len(df)
            print(f"成功读取CSV文件，共 {len(df)} 行数据")
            
            # 筛选offensive_label=1的数据
            filtered_df = df[df['offensive_label'] == offensive_label].copy()
            self.stats['filtered_rows'] = len(filtered_df)
            print(f"筛选到 offensive_label={offensive_label} 的数据 {len(filtered_df)} 行")
            
            # 提取需要的列
            required_columns = ['comment', 'rationales_annotator1', 'rationales_annotator2']
            
            # 检查列是否存在
            missing_columns = [col for col in required_columns if col not in df.columns]
            if missing_columns:
                print(f"警告: 以下列不存在: {missing_columns}")
                print(f"可用列: {list(df.columns)}")
                return None
            
            # 只保留需要的列
            result_df = filtered_df[required_columns].reset_index(drop=True)
            
            # 显示前几行数据预览
            print("\n=== 数据预览 (前5行) ===")
            for i, row in result_df.head(5).iterrows():
                print(f"行 {i}:")
                print(f"  comment: {row['comment'][:50]}...")
                print(f"  rationales_annotator1: {row['rationales_annotator1'][:50]}...")
                print(f"  rationales_annotator2: {row['rationales_annotator2'][:50]}...")
                print()
            
            return result_df
            
        except FileNotFoundError:
            print(f"错误: 找不到文件 {csv_path}")
            return None
        except Exception as e:
            print(f"读取CSV文件时发生错误: {e}")
            return None
    
    def clean_text(self, text):
        """清理文本，处理NaN值和特殊字符"""
        if pd.isna(text):
            return ""
        
        # 转换为字符串并清理
        text = str(text).strip()
        
        # 处理分号分隔的多个短语（保持原样，翻译时会整体翻译）
        return text
    
    def translate_column_data(self, df, columns_to_translate, batch_size=5, max_retries=3):
        """
        翻译指定列的数据
        
        参数:
            df: DataFrame数据
            columns_to_translate: 需要翻译的列名列表
            batch_size: 批量翻译大小
            max_retries: 最大重试次数
            
        返回:
            dict: 包含原始数据和翻译结果的字典
        """
        self.stats['start_time'] = datetime.now()
        results = {}
        
        for col in columns_to_translate:
            if col not in df.columns:
                print(f"警告: 列 '{col}' 不存在，跳过")
                continue
            
            print(f"\n开始翻译列: '{col}'...")
            original_data = df[col].apply(self.clean_text).tolist()
            translations = []
            failed_indices = []
            
            # 分批翻译
            for i in range(0, len(original_data), batch_size):
                batch_end = min(i + batch_size, len(original_data))
                batch = original_data[i:batch_end]
                
                # 显示进度
                progress = min(batch_end, len(original_data))
                progress_pct = progress / len(original_data) * 100
                print(f"  进度: {progress}/{len(original_data)} ({progress_pct:.1f}%)")
                
                # 处理当前批次
                for j, text in enumerate(batch):
                    idx = i + j
                    
                    if not text.strip():  # 空文本
                        translations.append("")
                        continue
                    
                    # 翻译当前文本（带重试）
                    translation, success = self._translate_with_retry(text, max_retries)
                    
                    if success:
                        translations.append(translation)
                        self.stats['translated_items'] += 1
                    else:
                        translations.append(f"[翻译失败] {text[:50]}")
                        failed_indices.append(idx)
                        self.stats['failed_items'] += 1
                
                # 添加延迟避免请求频率过高
                if batch_end < len(original_data):
                    time.sleep(0.2)
            
            # 保存结果
            results[f"{col}_original"] = original_data
            results[f"{col}_translated"] = translations
            
            if failed_indices:
                print(f"  列 '{col}' 有 {len(failed_indices)} 项翻译失败")
        
        self.stats['end_time'] = datetime.now()
        return results
    
    def _translate_with_retry(self, text, max_retries):
        """带重试机制的翻译"""
        for attempt in range(max_retries):
            try:
                translation = self.translator.translate(text)
                return translation, True
            except Exception as e:
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt  # 指数退避
                    time.sleep(wait_time)
                else:
                    print(f"翻译失败 (已重试{max_retries}次): {e}")
        
        return None, False
    
    def generate_bilingual_file(self, translation_results, output_file, format_type='detailed'):
        """
        生成双语对照文件
        
        参数:
            translation_results: 翻译结果字典
            output_file: 输出文件路径
            format_type: 输出格式 ('detailed'详细, 'compact'紧凑, 'csv'CSV格式)
        """
        try:
            # 确定有多少行数据
            first_key = list(translation_results.keys())[0]
            num_rows = len(translation_results[first_key])
            
            # 检查数据一致性
            for key in translation_results.keys():
                if len(translation_results[key]) != num_rows:
                    print(f"警告: 数据长度不一致: {key} 有 {len(translation_results[key])} 行")
            
            # 根据格式生成文件
            if format_type == 'csv':
                self._save_as_csv(translation_results, output_file)
            elif format_type == 'compact':
                self._save_compact_format(translation_results, output_file)
            else:  # 默认详细格式
                self._save_detailed_format(translation_results, output_file)
            
            print(f"\n✓ 双语对照文件已保存到: {output_file}")
            
        except Exception as e:
            print(f"生成双语文件时发生错误: {e}")
    
    def _save_detailed_format(self, results, output_file):
        """保存为详细格式"""
        # 获取列名
        columns = []
        for key in results.keys():
            if key.endswith('_original'):
                col_name = key.replace('_original', '')
                columns.append(col_name)
        
        # 确定行数
        first_key = list(results.keys())[0]
        num_rows = len(results[first_key])
        
        with open(output_file, 'w', encoding='utf-8') as f:
            # 文件头
            f.write("=" * 80 + "\n")
            f.write("葡萄牙语-中文双语对照翻译结果\n")
            f.write(f"源语言: {self.source_lang} -> 目标语言: {self.target_lang}\n")
            f.write(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"总数据行数: {num_rows}\n")
            f.write("=" * 80 + "\n\n")
            
            # 写入每行数据
            for row_idx in range(num_rows):
                f.write(f"【第 {row_idx+1} 行】\n")
                f.write("-" * 60 + "\n")
                
                for col in columns:
                    original_key = f"{col}_original"
                    translated_key = f"{col}_translated"
                    
                    if original_key in results and translated_key in results:
                        original_text = results[original_key][row_idx]
                        translated_text = results[translated_key][row_idx]
                        
                        # 写入列名和内容
                        f.write(f"{col.upper()}:\n")
                        f.write(f"  [PT] {original_text}\n")
                        f.write(f"  [CN] {translated_text}\n\n")
                
                f.write("=" * 80 + "\n\n")
    
    def _save_compact_format(self, results, output_file):
        """保存为紧凑格式"""
        # 获取列名
        columns = []
        for key in results.keys():
            if key.endswith('_original'):
                col_name = key.replace('_original', '')
                columns.append(col_name)
        
        # 确定行数
        first_key = list(results.keys())[0]
        num_rows = len(results[first_key])
        
        with open(output_file, 'w', encoding='utf-8') as f:
            # 文件头
            f.write("# 葡萄牙语-中文双语对照 (紧凑格式)\n")
            f.write(f"# 生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"# 总行数: {num_rows}\n\n")
            
            # 写入每行数据
            for row_idx in range(num_rows):
                f.write(f"行 {row_idx+1}:\n")
                
                for col in columns:
                    original_key = f"{col}_original"
                    translated_key = f"{col}_translated"
                    
                    if original_key in results and translated_key in results:
                        original_text = results[original_key][row_idx]
                        translated_text = results[translated_key][row_idx]
                        
                        # 写入列名和内容
                        f.write(f"  {col}: ")
                        f.write(f"{original_text} → {translated_text}\n")
                
                f.write("\n")
    
    def _save_as_csv(self, results, output_file):
        """保存为CSV格式"""
        # 获取列名
        columns = []
        for key in results.keys():
            if key.endswith('_original'):
                col_name = key.replace('_original', '')
                columns.append(col_name)
        
        # 确定行数
        first_key = list(results.keys())[0]
        num_rows = len(results[first_key])
        
        # 准备CSV数据
        csv_data = []
        header = []
        
        # 创建表头
        for col in columns:
            header.append(f"{col}_original")
            header.append(f"{col}_translated")
        
        csv_data.append(header)
        
        # 添加数据行
        for row_idx in range(num_rows):
            row_data = []
            for col in columns:
                original_key = f"{col}_original"
                translated_key = f"{col}_translated"
                
                if original_key in results and translated_key in results:
                    row_data.append(results[original_key][row_idx])
                    row_data.append(results[translated_key][row_idx])
                else:
                    row_data.append("")
                    row_data.append("")
            
            csv_data.append(row_data)
        
        # 保存为CSV
        import csv
        with open(output_file, 'w', encoding='utf-8', newline='') as f:
            writer = csv.writer(f)
            writer.writerows(csv_data)
    
    def show_statistics(self):
        """显示统计信息"""
        if self.stats['end_time'] and self.stats['start_time']:
            duration = (self.stats['end_time'] - self.stats['start_time']).total_seconds()
        else:
            duration = 0
        
        print("\n" + "=" * 60)
        print("翻译统计信息:")
        print("=" * 60)
        print(f"CSV总行数: {self.stats['total_rows']}")
        print(f"筛选后行数 (offensive_label=1): {self.stats['filtered_rows']}")
        print(f"成功翻译项: {self.stats['translated_items']}")
        print(f"翻译失败项: {self.stats['failed_items']}")
        if self.stats['translated_items'] + self.stats['failed_items'] > 0:
            success_rate = self.stats['translated_items'] / (self.stats['translated_items'] + self.stats['failed_items']) * 100
            print(f"翻译成功率: {success_rate:.1f}%")
        print(f"总耗时: {duration:.2f} 秒")
        if duration > 0 and self.stats['translated_items'] > 0:
            print(f"平均每项翻译时间: {duration/self.stats['translated_items']:.3f} 秒")
        print("=" * 60)


def main(sl,tl,file_name):
    """主函数 - 处理CSV文件翻译"""
    # 初始化翻译器
    translator = CSVTranslator(source_lang=sl, target_lang=tl)
    
    # CSV文件路径
    # csv_file = "HateBRXplain.csv"  # 替换为你的CSV文件名
    csv_file = file_name
    
    # 1. 加载并过滤CSV数据
    print("步骤1: 加载并过滤CSV数据...")
    filtered_data = translator.load_and_filter_csv(csv_file, offensive_label=1)
    
    if filtered_data is None or len(filtered_data) == 0:
        print("没有找到符合条件的行，程序退出。")
        return
    
    # 2. 翻译指定列
    print("\n步骤2: 开始翻译指定列...")
    columns_to_translate = ['comment', 'rationales_annotator1', 'rationales_annotator2']
    translation_results = translator.translate_column_data(
        filtered_data, 
        columns_to_translate, 
        batch_size=3,  # 小批量避免API限制
        max_retries=2
    )
    
    # 3. 生成双语对照文件
    print("\n步骤3: 生成双语对照文件...")
    output_file = "bilingual_translations.txt"
    translator.generate_bilingual_file(
        translation_results, 
        output_file,
        format_type='detailed'  # 可选: 'detailed', 'compact', 'csv'
    )
    
    # 4. 显示统计信息
    translator.show_statistics()
    
    # 5. 显示结果预览
    print("\n=== 翻译结果预览 (前3行) ===")
    for i in range(min(3, len(filtered_data))):
        print(f"\n第 {i+1} 行:")
        for col in columns_to_translate:
            original_key = f"{col}_original"
            translated_key = f"{col}_translated"
            
            if original_key in translation_results and translated_key in translation_results:
                print(f"  {col}:")
                print(f"    原文: {translation_results[original_key][i][:80]}...")
                print(f"    翻译: {translation_results[translated_key][i][:80]}...")


def simple_csv_translator():
    """简易版CSV翻译器"""
    # CSV文件路径
    csv_file = "comments.csv"
    
    try:
        # 读取CSV文件
        df = pd.read_csv(csv_file)
        print(f"成功读取CSV文件，共 {len(df)} 行")
        
        # 筛选offensive_label=1的数据
        filtered_df = df[df['offensive_label'] == 1].copy()
        print(f"筛选到 offensive_label=1 的数据 {len(filtered_df)} 行")
        
        # 只保留需要的列
        result_df = filtered_df[['comment', 'rationales_annotator1', 'rationales_annotator2']]
        
        # 初始化翻译器
        translator = GoogleTranslator(source='pt', target='zh-CN')
        
        # 准备存储翻译结果
        translations = {
            'comment_original': [],
            'comment_translated': [],
            'rationales_annotator1_original': [],
            'rationales_annotator1_translated': [],
            'rationales_annotator2_original': [],
            'rationales_annotator2_translated': []
        }
        
        # 逐行翻译
        for idx, row in result_df.iterrows():
            print(f"处理第 {idx+1}/{len(result_df)} 行...")
            
            # 翻译comment
            if pd.notna(row['comment']):
                try:
                    trans = translator.translate(str(row['comment']))
                    translations['comment_original'].append(str(row['comment']))
                    translations['comment_translated'].append(trans)
                except Exception as e:
                    translations['comment_original'].append(str(row['comment']))
                    translations['comment_translated'].append(f"[翻译失败: {e}]")
            
            # 翻译rationales_annotator1
            if pd.notna(row['rationales_annotator1']):
                try:
                    trans = translator.translate(str(row['rationales_annotator1']))
                    translations['rationales_annotator1_original'].append(str(row['rationales_annotator1']))
                    translations['rationales_annotator1_translated'].append(trans)
                except Exception as e:
                    translations['rationales_annotator1_original'].append(str(row['rationales_annotator1']))
                    translations['rationales_annotator1_translated'].append(f"[翻译失败: {e}]")
            
            # 翻译rationales_annotator2
            if pd.notna(row['rationales_annotator2']):
                try:
                    trans = translator.translate(str(row['rationales_annotator2']))
                    translations['rationales_annotator2_original'].append(str(row['rationales_annotator2']))
                    translations['rationales_annotator2_translated'].append(trans)
                except Exception as e:
                    translations['rationales_annotator2_original'].append(str(row['rationales_annotator2']))
                    translations['rationales_annotator2_translated'].append(f"[翻译失败: {e}]")
            
            # 添加延迟避免请求频率过高
            time.sleep(0.1)
        
        # 写入双语文件
        output_file = "simple_bilingual_output.txt"
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("葡萄牙语-中文双语翻译结果\n")
            f.write(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            # 确定有多少行数据
            num_rows = len(translations['comment_original'])
            
            for i in range(num_rows):
                f.write(f"【第 {i+1} 行】\n")
                f.write("-" * 60 + "\n")
                
                # comment
                f.write("COMMENT:\n")
                f.write(f"  [PT] {translations['comment_original'][i]}\n")
                f.write(f"  [CN] {translations['comment_translated'][i]}\n\n")
                
                # rationales_annotator1
                f.write("RATIONALE ANNOTATOR 1:\n")
                f.write(f"  [PT] {translations['rationales_annotator1_original'][i]}\n")
                f.write(f"  [CN] {translations['rationales_annotator1_translated'][i]}\n\n")
                
                # rationales_annotator2
                f.write("RATIONALE ANNOTATOR 2:\n")
                f.write(f"  [PT] {translations['rationales_annotator2_original'][i]}\n")
                f.write(f"  [CN] {translations['rationales_annotator2_translated'][i]}\n\n")
                
                f.write("=" * 60 + "\n\n")
        
        print(f"\n✓ 翻译完成！结果已保存到: {output_file}")
        
        # 显示预览
        print("\n=== 前3行结果预览 ===")
        for i in range(min(3, num_rows)):
            print(f"\n第 {i+1} 行:")
            print(f"  COMMENT: {translations['comment_original'][i][:50]}...")
            print(f"           → {translations['comment_translated'][i][:50]}...")
        
    except Exception as e:
        print(f"处理过程中发生错误: {e}")


def export_to_excel():
    """将翻译结果导出到Excel文件（包含原数据和翻译）"""
    csv_file = "comments.csv"
    
    try:
        # 读取CSV文件
        df = pd.read_csv(csv_file)
        
        # 筛选offensive_label=1的数据
        filtered_df = df[df['offensive_label'] == 1].copy()
        
        # 初始化翻译器
        translator = GoogleTranslator(source='pt', target='zh-CN')
        
        # 添加翻译列
        for col in ['comment', 'rationales_annotator1', 'rationales_annotator2']:
            print(f"正在翻译列: {col}...")
            
            translations = []
            for idx, text in enumerate(filtered_df[col]):
                if idx > 0 and idx % 10 == 0:
                    print(f"  进度: {idx}/{len(filtered_df)}")
                
                if pd.isna(text):
                    translations.append("")
                else:
                    try:
                        trans = translator.translate(str(text))
                        translations.append(trans)
                    except Exception as e:
                        translations.append(f"[翻译失败]")
                
                # 添加延迟
                time.sleep(0.1)
            
            filtered_df[f'{col}_cn'] = translations
        
        # 保存到Excel
        output_file = "translated_comments.xlsx"
        filtered_df.to_excel(output_file, index=False, encoding='utf-8')
        print(f"\n✓ Excel文件已保存到: {output_file}")
        
        # 保存到CSV（可选）
        csv_output = "translated_comments.csv"
        filtered_df.to_csv(csv_output, index=False, encoding='utf-8-sig')
        print(f"✓ CSV文件已保存到: {csv_output}")
        
    except Exception as e:
        print(f"导出Excel时发生错误: {e}")


# if __name__ == "__main__":
#     # 安装所需库（如果尚未安装）
#     # pip install pandas deep-translator openpyxl
    
#     print("CSV文件翻译工具")
#     print("=" * 50)
    
#     # 使用方法1：完整功能
#     main('id','zh-CN')
    
    # 使用方法2：简易版本
    # simple_csv_translator()
    
    # 使用方法3：导出到Excel
    # export_to_excel()

In [ ]:
def main(sl, tl, file_name):
    """主函数 - 处理CSV文件翻译（适配 Label, Tweet 格式）"""
    # 初始化翻译器
    translator = CSVTranslator(source_lang=sl, target_lang=tl)
    
    # 读取CSV文件（不筛选，直接全部加载）
    try:
        df = pd.read_csv(file_name)
        translator.stats['total_rows'] = len(df)
        print(f"成功读取CSV文件，共 {len(df)} 行数据")
        
        # 检查是否存在所需列
        required_columns = ['Label', 'Tweet']   # 根据实际列名修改
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            print(f"错误: 以下列不存在: {missing_columns}")
            print(f"可用列: {list(df.columns)}")
            return
        
        # 直接使用全部数据，不作筛选
        filtered_data = df[required_columns].reset_index(drop=True)
        translator.stats['filtered_rows'] = len(filtered_data)
        print(f"将翻译全部 {len(filtered_data)} 行数据")
        
        # 预览
        print("\n=== 数据预览 (前5行) ===")
        for i, row in filtered_data.head(5).iterrows():
            print(f"行 {i}:")
            print(f"  Label: {row['Label']}")
            print(f"  Tweet: {row['Tweet'][:50]}...")
            print()
            
    except FileNotFoundError:
        print(f"错误: 找不到文件 {file_name}")
        return
    except Exception as e:
        print(f"读取CSV文件时发生错误: {e}")
        return
    
    # 翻译指定列（这里需要翻译 Tweet 列，Label 一般不需要翻译）
    columns_to_translate = ['Tweet']   # 只翻译 Tweet 列
    translation_results = translator.translate_column_data(
        filtered_data, 
        columns_to_translate, 
        batch_size=3,
        max_retries=2
    )
    
    # 生成双语对照文件
    output_file = "bilingual_translations.txt"
    translator.generate_bilingual_file(
        translation_results, 
        output_file,
        format_type='detailed'
    )
    
    translator.show_statistics()
    
    # 预览结果
    print("\n=== 翻译结果预览 (前3行) ===")
    for i in range(min(3, len(filtered_data))):
        print(f"\n第 {i+1} 行:")
        print(f"  Label: {filtered_data.loc[i, 'Label']}")
        print(f"  原文: {translation_results['Tweet_original'][i][:80]}...")
        print(f"  翻译: {translation_results['Tweet_translated'][i][:80]}...")
main('id', 'zh-CN', 're_dataset_three_labels.csv')

In [10]:
import pandas as pd
import time
from datetime import datetime
from deep_translator import GoogleTranslator

def translate_label_tweet_batch(csv_path, source_lang='id', target_lang='zh-CN', batch_size=20, output_csv='translated_label_tweet.csv', output_txt='bilingual_label_tweet.txt'):
    """
    翻译格式为 Label,Tweet 的 CSV 文件，每翻译 batch_size 条就写入一次
    
    参数:
        csv_path: 输入 CSV 文件路径
        source_lang: 源语言代码 (默认 'id' 印度尼西亚语)
        target_lang: 目标语言代码 (默认 'zh-CN' 简体中文)
        batch_size: 每批次写入的数据量 (默认 20)
        output_csv: 输出 CSV 文件名
        output_txt: 输出双语文本文件名
    """
    # 读取 CSV
    try:
        df = pd.read_csv(csv_path)
        print(f"成功读取 {csv_path}，共 {len(df)} 行数据")
    except Exception as e:
        print(f"读取文件失败: {e}")
        return

    # 检查列名（注意大小写）
    if 'Label' not in df.columns or 'Tweet' not in df.columns:
        print("错误：CSV 文件必须包含 'Label' 和 'Tweet' 列")
        return

    # 初始化翻译器
    translator = GoogleTranslator(source=source_lang, target=target_lang)

    # 准备输出文件（先清空，若已有同名文件则覆盖）
    with open(output_csv, 'w', encoding='utf-8-sig') as f_csv:
        f_csv.write("Label,Tweet,Tweet_zh\n")   # 写表头
    with open(output_txt, 'w', encoding='utf-8') as f_txt:
        f_txt.write(f"印度尼西亚语 → 中文 双语对照\n生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

    # 统计数据
    total_rows = len(df)
    translated_count = 0
    failed_count = 0

    # 分批处理
    for start in range(0, total_rows, batch_size):
        end = min(start + batch_size, total_rows)
        batch_df = df.iloc[start:end].copy()   # 当前批次数据
        print(batch_df)

        print(f"\n正在处理第 {start+1} - {end} 行...")

        # 对当前批次进行翻译
        translations = []
        for idx, row in batch_df.iterrows():
            print(idx,row)
            text = row['Tweet']
            if pd.isna(text) or str(text).strip() == '':
                translations.append("")
                continue

            # 带重试的翻译
            success = False
            for attempt in range(3):
                try:
                    trans = translator.translate(str(text))
                    print(text)
                    print(trans)
                    translations.append(trans)
                    success = True
                    break
                except Exception as e:
                    if attempt < 2:
                        time.sleep(2 ** attempt)   # 指数退避
                    else:
                        print(f"  第 {idx+1} 行翻译失败: {e}")
                        translations.append("[翻译失败]")
            if success:
                translated_count += 1
            else:
                failed_count += 1

            # 每翻译一条稍作延迟，避免请求过快
            time.sleep(0.1)

        # 将翻译结果添加到批次 DataFrame
        batch_df['Tweet_zh'] = translations

        # 追加写入 CSV（模式 'a'，不写表头）
        batch_df[['Label', 'Tweet', 'Tweet_zh']].to_csv(
            output_csv, mode='a', header=False, index=False, encoding='utf-8-sig'
        )

        # 追加写入双语文本文件
        with open(output_txt, 'a', encoding='utf-8') as f_txt:
            for _, row in batch_df.iterrows():
                f_txt.write(f"行 {_+1}:\n")
                f_txt.write(f"  Label: {row['Label']}\n")
                f_txt.write(f"  [ID] {row['Tweet']}\n")
                f_txt.write(f"  [CN] {row['Tweet_zh']}\n")
                f_txt.write("-" * 60 + "\n")

        # 显示当前批次进度
        print(f"  本批次翻译完成，成功 {len(batch_df) - (translations.count('[翻译失败]') if '[翻译失败]' in translations else 0)} 条")
        print(f"  累计已写入 {end} 行到文件")

    # 最终统计
    print("\n" + "=" * 60)
    print("翻译完成！")
    print(f"总行数: {total_rows}")
    print(f"成功翻译: {translated_count}")
    print(f"失败: {failed_count}")
    print(f"CSV 文件: {output_csv}")
    print(f"双语文本: {output_txt}")
    print("=" * 60)


if __name__ == "__main__":
    # 调用示例
    translate_label_tweet_batch(
        csv_path='re_dataset_three_labels.csv',      # 替换为你的 CSV 文件名
        source_lang='id',
        target_lang='zh-CN',
        batch_size=20                  # 每 20 条写入一次
    )

成功读取 re_dataset_three_labels.csv，共 2016 行数据
    Label                                              Tweet
0       2  - Dia sendiri yang ngiklanin promo cashback di...
1       3  - disaat semua cowok berusaha melacak perhatia...
2       2  - kampret kan kalo typo-nya di email kantor ke...
3       2  - Mending makan disini lebih murah, buang-buan...
4       2  /biarin oppa masukim vibrator ke memek/ oppa k...
5       2  /peluk badan lu dr belakang; selipin tngan ked...
6       2  ada bumil ga? Jadi pengen ngentotin memek bumi...
7       2  ada perek yang imagine nya liar? Yg longrep ga...
8       2  ada perek yang mau dimainin memeknya? Yang lon...
9       2  cari yeoja yg chara seksi lonte, suka spam pic...
10      2  cowo yang jago drasex mana? sini ngentot sama ...
11      2  daritadi nungging sambil mainin memek sange bg...
12      2            duh rindu gesekan kontol doi ke memek~'
13      2             jepitin kontol gua pake memek lu sini'
14      2  lewat kamar kamu liat kamu elu

KeyboardInterrupt: 

In [3]:
"""
测试 OpenAI o3-mini-low 模型
"""

import os
from openai import OpenAI

# ==================== 配置 ====================
# 请替换为你的实际 API Key
API_KEY = "sk-JXgZv4AzWNPDKuiXE9ixc0w9mBJdkJoGJz19SBjy6bn0aY6p"

# 可选：使用代理或自定义 API 地址（如无特殊需求，使用默认 OpenAI 地址）
BASE_URL = "https://yinli.one/v1"

# 模型名称（根据 OpenAI 官方命名，目前为 "o3-mini"）
MODEL_NAME = "gpt-4o"  # 或 "o3-mini-2025-01-31" 等

# 测试消息
TEST_MESSAGE = "Hello, please introduce yourself briefly in one sentence."
# ==============================================

def test_o3_mini():
    """测试 o3-mini-low 模型（实际对应 o3-mini 的低推理强度版本）"""
    try:
        # 初始化客户端
        client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
        
        # 发送请求
        print(f"Sending test message to {MODEL_NAME}...")
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "user", "content": TEST_MESSAGE}
            ],
            # o3-mini 支持推理强度参数 reasoning_effort（可选 low, medium, high）
            # 这里显式设置为 low，对应 "o3-mini-low"
            reasoning_effort="low",
            max_completion_tokens=500,
            temperature=1.0
        )
        
        # 获取回复内容
        reply = response.choices[0].message.content
        print("\n=== Model Response ===")
        print(reply)
        print("\n=== Usage ===")
        print(f"Prompt tokens: {response.usage.prompt_tokens}")
        print(f"Completion tokens: {response.usage.completion_tokens}")
        print(f"Total tokens: {response.usage.total_tokens}")
        
    except Exception as e:
        print(f"Error occurred: {e}")

if __name__ == "__main__":
    test_o3_mini()

Sending test message to gpt-4o...
Error occurred: Error code: 503 - {'error': {'message': 'system memory overloaded', 'type': 'new_api_error', 'param': '', 'code': 'system_memory_overloaded'}}


In [6]:
from openai import OpenAI

# 初始化客户端（替换成你的第三方API地址和Key）
client = OpenAI(
    base_url="https://yinli.one/v1",
    api_key="sk-RTwmyPPqIe7YbZEyktRFozko2IaUQ3JGbZkmm614Ub7LerGY"
)

# 发送请求
completion = client.chat.completions.create(
    model="claude-3-5-haiku-20241022",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello!"}
    ]
)

# 打印返回结果
print(completion.choices[0].message)

InternalServerError: Error code: 503 - {'error': {'message': 'system memory overloaded', 'type': 'new_api_error', 'param': '', 'code': 'system_memory_overloaded'}}

In [ ]:
"""
使用大模型 API 对 CSV 中的文本进行多标签分类标注（7 个类别）。
支持异步并发、断点续传、批量写入。
所有配置请在 main() 函数中修改。
"""

import asyncio
import json
import os
import sys
from typing import List, Dict, Any

import pandas as pd
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm


# ==================== 辅助函数（无需修改） ====================
def get_system_prompt(country: str) -> str:
    """根据国家返回对应的系统提示词（7 分类，输出 labels 向量）"""
    base_prompt = """You are a content moderation expert. Analyze the given text and determine which of the following 7 categories it violates (multiple allowed). Output a JSON with "labels" (list of 7 integers, 0/1) and "reasoning" (short explanation).

Fixed category order:
1. Hate Speech & Group Attack
2. False & Misleading Information
3. Violence & Incitement to Violence
4. Harassment, Bullying & Privacy Violations
5. Sexual & Intimate Content
6. Illegal Transactions & Cybercrime
7. National Security & Public Order

Output format example:
{"labels": [1,0,0,1,0,0,0], "reasoning": "The text contains hate speech and doxxing."}
If no violation, output all zeros: {"labels": [0,0,0,0,0,0,0], "reasoning": "Safe content."}
Only output valid JSON, no extra text.
"""
    if country == "Singapore":
        additional = """
Special focus for Singapore (OSRA Act): 
- Hate speech includes incitement of enmity based on race/religion.
- Harassment includes doxxing, online stalking, impersonation, defamation.
- False information that harms public health/safety is critical.
- Violence incitement is strictly forbidden.
"""
    elif country == "Indonesia":
        additional = """
Special focus for Indonesia (UU ITE, SARA):
- Hate speech targeting ethnicity, religion, race, inter-group (SARA) is most sensitive.
- False information (hoax) that causes public panic is aggressively removed.
- Pornography and LGBTQ+ content may be considered obscene.
- Gambling promotion is illegal.
"""
    elif country == "Thailand":
        additional = """
Special focus for Thailand (CCA, Lèse Majesté):
- Any negative comment about the monarchy (king, queen, heir) is a serious crime (Article 112).
- False information that harms national security or causes public panic is prohibited.
- Obscene content and gambling are illegal.
- Criticism of military or government may threaten public order.
"""
    else:
        additional = ""
    return base_prompt + additional


def load_progress(output_csv: str) -> int:
    """读取已标注的行数（假设输出文件已存在且保留了原始顺序）"""
    if os.path.exists(output_csv):
        try:
            df_done = pd.read_csv(output_csv)
            return len(df_done)
        except Exception:
            return 0
    return 0


def save_batch(df_original: pd.DataFrame, start_idx: int, end_idx: int, results: List[Dict], output_csv: str):
    """
    将一批标注结果合并到原始数据，并写入文件。
    如果文件已存在则追加，否则新建。
    结果中包含原始所有列 + labels + reasoning。
    """
    batch_original = df_original.iloc[start_idx:end_idx].copy()
    batch_original["labels"] = [r.get("labels", "[0,0,0,0,0,0,0]") for r in results]
    batch_original["reasoning"] = [r.get("reasoning", "") for r in results]
    
    if not os.path.exists(output_csv) or start_idx == 0:
        batch_original.to_csv(output_csv, index=False, mode='w')
    else:
        batch_original.to_csv(output_csv, index=False, mode='a', header=False)


async def call_api(
    client: AsyncOpenAI,
    system_prompt: str,
    user_text: str,
    semaphore: asyncio.Semaphore,
    model_name: str,
    max_retries: int = 3,        # 改为整数，默认3
    retry_delay: float = 2.0
) -> Dict[str, Any]:
    """单次 API 调用，带重试和并发控制"""
    async with semaphore:
        for attempt in range(max_retries):   # 现在 max_retries 是整数，没问题
            try:
                response = await client.chat.completions.create(
                    model=model_name,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_text}
                    ],
                    temperature=0.0,
                    max_tokens=500,
                    response_format={"type": "json_object"}
                )
                content = response.choices[0].message.content
                parsed = json.loads(content)
                if "labels" not in parsed:
                    parsed["labels"] = [0]*7
                if "reasoning" not in parsed:
                    parsed["reasoning"] = ""
                if len(parsed["labels"]) != 7:
                    parsed["labels"] = [0]*7
                parsed["labels"] = [int(x) for x in parsed["labels"]]
                return parsed
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"API call failed after {max_retries} attempts: {e}")
                    return {"labels": [0]*7, "reasoning": f"API error: {str(e)}"}
                await asyncio.sleep(retry_delay * (attempt + 1))
        return {"labels": [0]*7, "reasoning": "Unknown error"}


async def process_batch(
    client: AsyncOpenAI,
    df: pd.DataFrame,
    start_idx: int,
    end_idx: int,
    system_prompt: str,
    semaphore: asyncio.Semaphore,
    text_col: str,
    model_name: str,
    max_retries: int = 3,
    retry_delay: float = 2.0
) -> List[Dict]:
    """处理一批文本（异步并发）"""
    tasks = []
    for idx in range(start_idx, end_idx):
        text = str(df.iloc[idx][text_col])
        if not text or text == "nan":
            text = ""
        # 使用闭包传递参数
        async def single_call(t=text):
            return await call_api(
                client, system_prompt, t, semaphore,
                model_name=model_name,
                max_retries=max_retries,
                retry_delay=retry_delay
            )
        tasks.append(single_call())
    results = await tqdm.gather(*tasks, desc=f"Processing rows {start_idx}-{end_idx-1}")
    return results


# ==================== 主函数 ====================
async def main():
    # ========== 在这里修改你的配置 ==========
    # 文件路径
    INPUT_CSV = "input.csv"           # 原始数据文件
    OUTPUT_CSV = "output_labeled.csv" # 标注结果文件（自动创建/续写）
    
    # 数据列名
    TEXT_COL = "text"                 # 待标注的文本所在的列名
    
    # 大模型 API 配置
    API_BASE = "https://api.openai.com/v1"
    API_KEY = "your-api-key-here"
    MODEL_NAME = "gpt-4o-mini"        # 或 "o3-mini" 等
    
    # 请求参数
    MAX_CONCURRENT = 10               # 最大并发请求数（整数）
    BATCH_SIZE = 20                   # 每处理多少条写入一次文件（整数）
    MAX_RETRIES = 3                   # 最大重试次数（整数）
    RETRY_DELAY = 2.0                 # 重试延迟（浮点数）
    
    # 国家/地区（用于提示词，可选 "Singapore", "Indonesia", "Thailand"）
    COUNTRY = "Singapore"
    
    # 可选：完全自定义系统提示词（如果不为空，则忽略 COUNTRY）
    CUSTOM_SYSTEM_PROMPT = None
    # ======================================
    
    # 读取原始 CSV
    if not os.path.exists(INPUT_CSV):
        print(f"Input file not found: {INPUT_CSV}")
        sys.exit(1)
    df = pd.read_csv(INPUT_CSV)
    total_rows = len(df)
    print(f"Total rows: {total_rows}")
    
    # 确定已处理行数
    processed = load_progress(OUTPUT_CSV)
    if processed >= total_rows:
        print("All rows already labeled. Exiting.")
        return
    print(f"Resuming from row {processed} (already labeled {processed} rows)")
    
    # 初始化 OpenAI 客户端
    client = AsyncOpenAI(base_url=API_BASE, api_key=API_KEY)
    
    # 确定系统提示词
    if CUSTOM_SYSTEM_PROMPT:
        system_prompt = CUSTOM_SYSTEM_PROMPT
    else:
        system_prompt = get_system_prompt(COUNTRY)
    print("Using system prompt:\n", system_prompt[:200], "...\n")
    
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    
    # 分批处理
    for batch_start in range(processed, total_rows, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, total_rows)
        print(f"\nProcessing batch {batch_start}-{batch_end-1}...")
        results = await process_batch(
            client, df, batch_start, batch_end, system_prompt, semaphore,
            text_col=TEXT_COL,
            model_name=MODEL_NAME,
            max_retries=MAX_RETRIES,
            retry_delay=RETRY_DELAY
        )
        save_batch(df, batch_start, batch_end, results, OUTPUT_CSV)
        print(f"Batch saved to {OUTPUT_CSV}")
    
    print(f"\nAll done. Labeled file saved to {OUTPUT_CSV}")


if __name__ == "__main__":
    asyncio.run(main())

ython3 new_processing_dataset.py -i new_dataset.csv -o gemini-2.5-flash_output.csv -m gemini-2.5-flash --api-base https://yinli.one/v1 --api-key sk-RTwmyPPqIe7YbZEyktRFozko2IaUQ3JGbZkmm614Ub7LerGY